# GNN-CNN Hybrid Channel Prediction for Cell-Free Massive MIMO

**Research area:** AI-Native Physical Layer Design for 6G  
**Problem:** Multi-step forward channel state prediction in distributed cell-free massive MIMO under high user mobility
#
This notebook runs the entire pipeline:
1. Generate synthetic channel datasets (Jake's + Gudmundson model)
2. Train GNN-CNN and CNN-Only models across all velocity × layout configs
3. Evaluate all experiments and produce plots

## 0. Setup & Imports

In [1]:
import os
import time
import json
import argparse
import numpy as np
import torch
import torch.nn as nn
from torch.utils.data import TensorDataset, DataLoader, random_split
from scipy.special import j0
from typing import Tuple

import matplotlib
matplotlib.use('Agg')
import matplotlib.pyplot as plt

DEVICE = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print(f"Device: {DEVICE}")

# Paths
DATA_DIR    = '/kaggle/working/data'
CKPT_DIR    = '/kaggle/working/checkpoints'
RESULTS_DIR = '/kaggle/working/results'
os.makedirs(DATA_DIR, exist_ok=True)
os.makedirs(CKPT_DIR, exist_ok=True)
os.makedirs(RESULTS_DIR, exist_ok=True)

Device: cuda


## 1. Dataset Generation (dataset.py)

Channel model:
- Large-scale fading: 3GPP UMa path loss + spatially correlated shadow fading
- Small-scale fading: Jake's isotropic scattering (AR(1) process)
- Spatial AP correlation via Gudmundson model: C_ll' = 2^(-d_ll' / d_decorr)

In [2]:
# ── AP / UE geometry ──

def generate_ap_positions(L, area_size=500.0, layout='random', seed=42):
    """Return AP positions (L, 2) in metres."""
    rng = np.random.RandomState(seed)
    if layout == 'random':
        return rng.uniform(0, area_size, (L, 2))
    elif layout == 'grid':
        g = int(np.sqrt(L))
        assert g * g == L, "L must be a perfect square for grid layout"
        xs = np.linspace(area_size / (2 * g), area_size - area_size / (2 * g), g)
        ys = np.linspace(area_size / (2 * g), area_size - area_size / (2 * g), g)
        xx, yy = np.meshgrid(xs, ys)
        return np.column_stack([xx.flatten(), yy.flatten()])
    else:
        raise ValueError(f"Unknown layout: {layout}")


def generate_ue_positions(K, area_size=500.0, seed=0):
    """Return UE positions (K, 2) in metres."""
    rng = np.random.RandomState(seed)
    margin = 0.1 * area_size
    return rng.uniform(margin, area_size - margin, (K, 2))


# ── Large-scale fading with spatial correlation ──

def _path_loss_uma(d_m, fc_ghz=3.5):
    """3GPP UMa path loss (linear scale)."""
    d_m = np.maximum(d_m, 10.0)
    PL_dB = 128.1 + 37.6 * np.log10(d_m / 1000.0)
    return 10.0 ** (-PL_dB / 10.0)


def _shadow_corr_matrix(ap_pos, d_decorr=9.0):
    """Gudmundson inter-AP shadow fading correlation matrix (L, L)."""
    L = len(ap_pos)
    diff = ap_pos[:, None, :] - ap_pos[None, :, :]
    dist = np.linalg.norm(diff, axis=-1)
    C = np.power(2.0, -dist / d_decorr)
    C += 1e-6 * np.eye(L)
    return C


def generate_large_scale_fading(ap_pos, ue_pos, fc_ghz=3.5,
                                 shadow_std_db=8.0, seed=0):
    """Generate beta_lk = PL_lk * 10^(z_lk / 10), shape (L, K)."""
    L, K = len(ap_pos), len(ue_pos)
    rng = np.random.RandomState(seed)

    beta = np.zeros((L, K))
    for k in range(K):
        d = np.linalg.norm(ap_pos - ue_pos[k], axis=1)
        beta[:, k] = _path_loss_uma(d, fc_ghz)

    C = _shadow_corr_matrix(ap_pos)
    try:
        L_chol = np.linalg.cholesky(C)
    except np.linalg.LinAlgError:
        C += 1e-4 * np.eye(L)
        L_chol = np.linalg.cholesky(C)

    for k in range(K):
        z_raw = rng.randn(L)
        z_db = shadow_std_db * (L_chol @ z_raw)
        beta[:, k] *= 10.0 ** (z_db / 10.0)

    return beta


# ── Temporal channel (Jake's AR(1) model) ──

def generate_temporal_channel(beta, ap_pos, ue_pos, N, T_total, fc, velocity_kmh,
                               Ts=1e-3, num_clusters=20, seed=0):
    """
    Geometry-Based Stochastic Model (GBSM) approximating 3GPP/WINNER II channel models.
    Generates realistic spatial correlation across both distributed APs and local AP antennas.
    """
    rng = np.random.RandomState(seed)
    L, K = beta.shape
    c_speed = 3e8
    lam = c_speed / fc
    
    # 1. 2D Environment setup - Place scattering clusters randomly
    area_min = min(ap_pos.min(), ue_pos.min()) - 100
    area_max = max(ap_pos.max(), ue_pos.max()) + 100
    cluster_pos = rng.uniform(area_min, area_max, (num_clusters, 2))
    
    # 2. UE velocities (random directions for each user)
    v_ms = velocity_kmh / 3.6
    angles = rng.uniform(0, 2 * np.pi, K)
    v_vec = np.stack([np.cos(angles), np.sin(angles)], axis=1) * v_ms  # (K, 2)
    
    # 3. Base complex scattering amplitude for each cluster -> UE link
    A = (rng.randn(num_clusters, K) + 1j * rng.randn(num_clusters, K)) / np.sqrt(2)
    
    H = np.zeros((T_total, L, N, K), dtype=complex)
    
    # Distance: Cluster to APs (Fixed over time)
    diff_c_ap = cluster_pos[:, None, :] - ap_pos[None, :, :]
    d_c_ap = np.linalg.norm(diff_c_ap, axis=-1)  # (C, L)
    
    # Angle of Arrival (AoA) at the AP from the cluster.
    aoa_c_ap = np.arctan2(diff_c_ap[:, :, 1], diff_c_ap[:, :, 0]) # (C, L)
    
    # ULA array steering vectors for N antennas (spacing = lambda/2)
    antenna_spacing = lam / 2.0
    n_idx = np.arange(N)
    
    for t in range(T_total):
        # Move UEs geometrically
        ue_pos_t = ue_pos + v_vec * (t * Ts)  # (K, 2)
        
        # Distance: UE to Cluster (Time-varying) -> generates Doppler physically!
        diff_ue_c = ue_pos_t[None, :, :] - cluster_pos[:, None, :]  # (C, K, 2)
        d_ue_c = np.linalg.norm(diff_ue_c, axis=-1)  # (C, K)
        
        # Total propagation distance: D_clk(t) = d(UE, Cluster) + d(Cluster, AP)
        D = d_c_ap[:, :, None] + d_ue_c[:, None, :]  # (C, L, K)
        
        # Base phase from path length
        phase_base = -2j * np.pi * D / lam  # (C, L, K)
        
        # Array Response Phase Offset
        array_phase = -2j * np.pi * (antenna_spacing / lam) * n_idx[None, None, :] * np.cos(aoa_c_ap)[:, :, None]
        
        # Total complex phase (C, L, N, K)
        total_phase = phase_base[:, :, None, :] + array_phase[:, :, :, None]
        
        # Superimpose the arriving waves from all C clusters!
        h_t = np.sum(A[:, None, None, :] * np.exp(total_phase), axis=0)  # (L, N, K)
        
        # Normalize so variance is 1
        h_t /= np.sqrt(num_clusters)
        
        # Scale to match the large-scale envelope (pathloss + shadow fading)
        H[t] = h_t * np.sqrt(beta[:, None, :])
        
    return H


# ── Graph construction ──

def build_knn_graph(ap_pos, k_neighbors=3):
    """Build symmetric K-NN adjacency from AP positions."""
    L = len(ap_pos)
    diff = ap_pos[:, None, :] - ap_pos[None, :, :]
    dist = np.linalg.norm(diff, axis=-1)
    np.fill_diagonal(dist, np.inf)

    adj = np.zeros((L, L), dtype=np.float32)
    for i in range(L):
        nbrs = np.argsort(dist[i])[:k_neighbors]
        adj[i, nbrs] = 1.0
        adj[nbrs, i] = 1.0

    rows, cols = np.nonzero(adj)
    edge_index = torch.tensor(np.stack([rows, cols]), dtype=torch.long)
    return edge_index, adj


def compute_norm_adj(adj, self_loops=True):
    """Compute D^{-1/2} (A + I) D^{-1/2} for GCN propagation."""
    A = adj.copy().astype(np.float32)
    if self_loops:
        A += np.eye(len(A), dtype=np.float32)
    D = A.sum(axis=1)
    D_inv_sqrt = np.diag(D ** -0.5)
    A_norm = D_inv_sqrt @ A @ D_inv_sqrt
    return torch.tensor(A_norm, dtype=torch.float32)


# ── Full dataset generation ──

def generate_dataset(L=16, K=4, N=4, fc=3.5e9, velocity_kmh=60.0,
                     T_history=10, T_predict=3, num_samples=5000,
                     layout='random', area_size=500.0, ap_seed=42,
                     Ts=1e-3, save_path=None, verbose=True):
    """Generate full dataset."""
    ap_pos = generate_ap_positions(L, area_size, layout, seed=ap_seed)
    edge_index, adj_np = build_knn_graph(ap_pos, k_neighbors=3)
    adj_norm = compute_norm_adj(adj_np)

    T_total = T_history + T_predict
    C = 2 * N * K

    X_list, Y_list = [], []

    for s in range(num_samples):
        if verbose and s % 500 == 0:
            print(f"  Generating sample {s}/{num_samples}...")

        ue_pos = generate_ue_positions(K, area_size, seed=s)
        beta = generate_large_scale_fading(ap_pos, ue_pos,
                                            fc_ghz=fc / 1e9, seed=s)
        H = generate_temporal_channel(beta, ap_pos, ue_pos, N, T_total, fc,
                                       velocity_kmh, Ts, seed=s)

        H_r = H.real.reshape(T_total, L, N * K)
        H_i = H.imag.reshape(T_total, L, N * K)
        H_flat = np.concatenate([H_r, H_i], axis=-1).astype(np.float32)

        H_t = H_flat.transpose(1, 2, 0)

        X_list.append(H_t[:, :, :T_history])
        Y_list.append(H_t[:, :, T_history:])

    X = torch.tensor(np.stack(X_list))
    Y = torch.tensor(np.stack(Y_list))

    dataset = {
        'X': X, 'Y': Y,
        'edge_index': edge_index, 'adj_norm': adj_norm,
        'adj_np': adj_np,
        'ap_pos': torch.tensor(ap_pos, dtype=torch.float32),
        'layout': layout, 'velocity_kmh': velocity_kmh,
        'L': L, 'K': K, 'N': N,
        'T_history': T_history, 'T_predict': T_predict,
        'fc': fc, 'Ts': Ts,
    }

    if save_path:
        os.makedirs(os.path.dirname(os.path.abspath(save_path)), exist_ok=True)
        torch.save(dataset, save_path)
        if verbose:
            print(f"Saved -> {save_path}  |  X:{X.shape}  Y:{Y.shape}")

    return dataset


def generate_all_datasets(data_dir=DATA_DIR, **kwargs):
    """Pre-generate all required datasets."""
    os.makedirs(data_dir, exist_ok=True)
    velocities = [30, 60, 120]
    layouts = ['random', 'grid']

    for vel in velocities:
        for lay in layouts:
            path = os.path.join(data_dir, f'dataset_{lay}_{vel}kmh.pt')
            if os.path.exists(path):
                print(f"Already exists, skipping: {path}")
                continue
            print(f"\n-- Generating: layout={lay}, velocity={vel} km/h --")
            generate_dataset(velocity_kmh=vel, layout=lay,
                             save_path=path, **kwargs)

## 2. Model Architecture (model.py)
#
Architecture (GNNCNNHybrid):
- Stage 1 -- LocalCNN: Conv1D x 3 per AP (shared weights) -> h_loc in R^64
- Stage 2 -- GCNLayer: single normalised GCN -> h_agg in R^64
- Stage 3 -- PredHead: Linear(64->128->out) per AP -> (2*N*K, T_predict)

In [3]:
class LocalCNN(nn.Module):
    """Per-AP temporal feature extractor (shared weights across all APs)."""

    def __init__(self, in_channels, hidden_dim=64):
        super().__init__()
        self.net = nn.Sequential(
            nn.Conv1d(in_channels, 32, kernel_size=3, padding=1),
            nn.BatchNorm1d(32),
            nn.ReLU(),
            nn.Conv1d(32, 64, kernel_size=3, padding=1),
            nn.BatchNorm1d(64),
            nn.ReLU(),
            nn.Conv1d(64, 64, kernel_size=3, padding=1),
            nn.BatchNorm1d(64),
            nn.ReLU(),
            nn.AdaptiveAvgPool1d(1),
        )
        self.proj = nn.Sequential(
            nn.Linear(64, hidden_dim),
            nn.ReLU(),
        )

    def forward(self, x):
        h = self.net(x).squeeze(-1)
        return self.proj(h)


class GCNLayer(nn.Module):
    """Single GCN message-passing layer."""

    def __init__(self, in_features, out_features):
        super().__init__()
        self.linear = nn.Linear(in_features, out_features, bias=True)
        self.relu = nn.ReLU()

    def forward(self, h, adj_norm):
        support = self.linear(h)
        out = torch.einsum('lm,bmd->bld', adj_norm, support)
        return self.relu(out)


class PredictionHead(nn.Module):
    """Per-AP prediction MLP."""

    def __init__(self, hidden_dim, out_channels, T_predict):
        super().__init__()
        self.T_predict = T_predict
        self.out_channels = out_channels
        self.net = nn.Sequential(
            nn.Linear(hidden_dim, 128),
            nn.ReLU(),
            nn.Linear(128, out_channels * T_predict),
        )

    def forward(self, h):
        return self.net(h)


class GNNCNNHybrid(nn.Module):
    """Three-stage GNN-CNN hybrid for multi-step channel prediction."""

    def __init__(self, L, N, K, T_history, T_predict, hidden_dim=64):
        super().__init__()
        self.L = L
        self.T_predict = T_predict
        self.out_channels = 2 * N * K

        self.local_cnn = LocalCNN(in_channels=self.out_channels,
                                   hidden_dim=hidden_dim)
        self.gcn = GCNLayer(hidden_dim, hidden_dim)
        self.pred_head = PredictionHead(hidden_dim, self.out_channels, T_predict)

    def forward(self, x, adj_norm):
        B, L, C, T = x.shape
        
        # 1. Normalize per AP
        scale = torch.sqrt((x**2).mean(dim=(-1, -2), keepdim=True) + 1e-8)
        x_norm = x / scale

        x_flat = x_norm.reshape(B * L, C, T)
        h_loc = self.local_cnn(x_flat)
        h_loc = h_loc.reshape(B, L, -1)
        h_agg = self.gcn(h_loc, adj_norm)
        h_flat = h_agg.reshape(B * L, -1)
        pred_flat = self.pred_head(h_flat)
        pred_norm = pred_flat.reshape(B, L, self.out_channels, self.T_predict)
        
        # 2. Denormalize
        pred = pred_norm * scale
        return pred


class CNNOnly(nn.Module):
    """Ablation model: CNN per AP with NO graph aggregation."""

    def __init__(self, L, N, K, T_history, T_predict, hidden_dim=64):
        super().__init__()
        self.L = L
        self.T_predict = T_predict
        self.out_channels = 2 * N * K

        self.local_cnn = LocalCNN(in_channels=self.out_channels,
                                   hidden_dim=hidden_dim)
        self.pred_head = PredictionHead(hidden_dim, self.out_channels, T_predict)

    def forward(self, x, adj_norm=None):
        B, L, C, T = x.shape
        
        # 1. Normalize per AP
        scale = torch.sqrt((x**2).mean(dim=(-1, -2), keepdim=True) + 1e-8)
        x_norm = x / scale

        x_flat = x_norm.reshape(B * L, C, T)
        h_loc = self.local_cnn(x_flat)
        pred_flat = self.pred_head(h_loc)
        pred_norm = pred_flat.reshape(B, L, self.out_channels, self.T_predict)
        
        # 2. Denormalize
        pred = pred_norm * scale
        return pred


# ── Loss & Utilities ──

def nmse_loss(pred, target):
    """NMSE loss (linear) -- used during training."""
    num = ((pred - target) ** 2).sum(dim=(1, 2, 3))
    den = (target ** 2).sum(dim=(1, 2, 3)) + 1e-8
    return (num / den).mean()


def nmse_db(pred, target):
    """NMSE in dB -- used during evaluation."""
    with torch.no_grad():
        num = ((pred - target) ** 2).sum()
        den = (target ** 2).sum() + 1e-8
        return 10.0 * torch.log10(num / den).item()


def count_parameters(model):
    return sum(p.numel() for p in model.parameters() if p.requires_grad)


def measure_latency(model, x, adj_norm, n_runs=100):
    """Measure mean inference latency in ms."""
    model.eval()
    device = next(model.parameters()).device
    x = x.to(device)
    adj_norm = adj_norm.to(device)

    with torch.no_grad():
        for _ in range(10):
            _ = model(x, adj_norm)

    times = []
    with torch.no_grad():
        for _ in range(n_runs):
            if device.type == 'cuda':
                torch.cuda.synchronize()
            t0 = time.perf_counter()
            _ = model(x, adj_norm)
            if device.type == 'cuda':
                torch.cuda.synchronize()
            times.append((time.perf_counter() - t0) * 1000)

    return float(np.mean(times)), float(np.std(times))

## 3. Baselines (baselines.py)
#
- AR(4) predictor with Yule-Walker coefficient estimation
- Kalman filter: optimal linear predictor for Jake's AR(1) model

In [4]:
def _yule_walker(x, order):
    """Estimate AR coefficients via Yule-Walker equations."""
    n = len(x)
    r = np.array([np.dot(np.conj(x[:n - k]), x[k:]) / (n - k)
                  for k in range(order + 1)])
    R = np.array([[r[abs(i - j)] for j in range(order)]
                  for i in range(order)])
    try:
        a = np.linalg.solve(R, r[1:])
    except np.linalg.LinAlgError:
        a = np.zeros(order, dtype=complex)
    return a


def ar_predict(h_history, T_predict, order=4):
    """Predict T_predict future values using AR(P)."""
    P = min(order, len(h_history) - 1)
    if P < 1:
        return np.full(T_predict, h_history[-1], dtype=complex)

    coeffs = _yule_walker(h_history, P)
    buf = list(h_history[-P:])
    preds = []
    for _ in range(T_predict):
        pred = np.dot(coeffs, buf[-P:][::-1])
        preds.append(pred)
        buf.append(pred)
    return np.array(preds)


class ARPredictor:
    def __init__(self, order=4):
        self.order = order

    def predict(self, X, T_predict):
        L, N, K, T_h = X.shape
        Y = np.zeros((L, N, K, T_predict), dtype=complex)
        for l in range(L):
            for n in range(N):
                for k in range(K):
                    Y[l, n, k] = ar_predict(X[l, n, k], T_predict, self.order)
        return Y


def _estimate_ar_scalar_joint(X_seq, p=1):
    N, T = X_seq.shape
    if T <= p:
        return np.zeros(p, dtype=complex)
    
    r = np.zeros(p + 1, dtype=complex)
    for tau in range(p + 1):
        for n in range(N):
            r[tau] += np.dot(np.conj(X_seq[n, :T - tau]), X_seq[n, tau:]) / (T - tau)
    r /= N
    
    R = np.zeros((p, p), dtype=complex)
    for i in range(p):
        for j in range(p):
            R[i, j] = r[abs(i - j)] if i >= j else np.conj(r[abs(i - j)])
            
    try:
        a = np.linalg.solve(R, r[1:])
    except np.linalg.LinAlgError:
        a = np.zeros(p, dtype=complex)
    return a


class VectorKalmanPredictor:
    def __init__(self, order=1):
        self.p = order

    def predict(self, X, T_predict):
        L, N, K, T_h = X.shape
        Y = np.zeros((L, N, K, T_predict), dtype=complex)
        
        for l in range(L):
            for k in range(K):
                h_seq = X[l, :, k, :]
                p = min(self.p, T_h - 1)
                
                if p < 1:
                    Y[l, :, k, :] = h_seq[:, -1:]
                    continue
                    
                a = _estimate_ar_scalar_joint(h_seq, p)
                
                if T_h > p:
                    W = np.zeros((N, T_h - p), dtype=complex)
                    for t in range(p, T_h):
                        h_t = h_seq[:, t]
                        h_pred = np.zeros(N, dtype=complex)
                        for i in range(p):
                            h_pred += a[i] * h_seq[:, t - 1 - i]
                        W[:, t - p] = h_t - h_pred
                    Q = (W @ np.conj(W).T) / (T_h - p)
                else:
                    Q = np.eye(N, dtype=complex) * 1e-6
                
                Q += 1e-6 * np.eye(N)
                R_obs = Q * 0.1
                
                S = np.zeros(p * N, dtype=complex)
                for i in range(p):
                    S[i*N : (i+1)*N] = h_seq[:, p - 1 - i]
                
                P = np.eye(p * N, dtype=complex) * np.mean(np.diag(np.abs(Q)))
                
                F = np.zeros((p * N, p * N), dtype=complex)
                for i in range(p):
                    F[0:N, i*N : (i+1)*N] = a[i] * np.eye(N)
                for i in range(1, p):
                    F[i*N : (i+1)*N, (i-1)*N : i*N] = np.eye(N)
                    
                Q_ss = np.zeros((p * N, p * N), dtype=complex)
                Q_ss[0:N, 0:N] = Q
                
                H_obs = np.zeros((N, p * N), dtype=complex)
                H_obs[0:N, 0:N] = np.eye(N)
                
                for t in range(p, T_h):
                    y_t = h_seq[:, t]
                    
                    S_pred = F @ S
                    P_pred = F @ P @ np.conj(F).T + Q_ss
                    
                    Inn = y_t - (H_obs @ S_pred)
                    S_cov = H_obs @ P_pred @ np.conj(H_obs).T + R_obs
                    try:
                        K_gain = P_pred @ np.conj(H_obs).T @ np.linalg.inv(S_cov)
                    except np.linalg.LinAlgError:
                        K_gain = np.zeros((p*N, N), dtype=complex)
                        
                    S = S_pred + K_gain @ Inn
                    P = (np.eye(p * N) - K_gain @ H_obs) @ P_pred
                    
                for step in range(T_predict):
                    S = F @ S
                    Y[l, :, k, step] = S[0:N]
                    
        return Y


def nmse_complex(pred, target):
    """NMSE in dB for complex-valued arrays."""
    num = np.sum(np.abs(pred - target) ** 2)
    den = np.sum(np.abs(target) ** 2) + 1e-12
    return 10.0 * np.log10(num / den)


def evaluate_baseline(predictor, X_complex, Y_complex, T_predict, k_step=None):
    S = X_complex.shape[0]
    preds = []
    for s in range(S):
        pred_s = predictor.predict(X_complex[s], T_predict)
        preds.append(pred_s)
    preds = np.stack(preds)

    if k_step is not None:
        return nmse_complex(preds[..., k_step], Y_complex[..., k_step])
    return nmse_complex(preds, Y_complex)


def flat_to_complex(X_flat, N, K):
    """Convert (S, L, 2*N*K, T) -> (S, L, N, K, T) complex."""
    X_np = X_flat.numpy()
    S, L, C, T = X_np.shape
    NK = N * K
    X_r = X_np[:, :, :NK, :].reshape(S, L, N, K, T)
    X_i = X_np[:, :, NK:, :].reshape(S, L, N, K, T)
    return X_r + 1j * X_i

## 4. Training (train.py)

In [5]:
def load_or_generate(layout, velocity, data_dir=DATA_DIR, **kwargs):
    path = os.path.join(data_dir, f'dataset_{layout}_{int(velocity)}kmh.pt')
    if os.path.exists(path):
        print(f"Loading dataset from {path}")
        return torch.load(path, weights_only=False)
    print(f"Dataset not found at {path}, generating...")
    os.makedirs(data_dir, exist_ok=True)
    return generate_dataset(layout=layout, velocity_kmh=velocity,
                             save_path=path, **kwargs)


def make_loaders(dataset, batch_size=32, train_frac=0.8, val_frac=0.1, seed=0):
    X, Y = dataset['X'], dataset['Y']
    full = TensorDataset(X, Y)
    S = len(full)
    n_train = int(S * train_frac)
    n_val = int(S * val_frac)
    n_test = S - n_train - n_val

    gen = torch.Generator().manual_seed(seed)
    train_ds, val_ds, test_ds = random_split(
        full, [n_train, n_val, n_test], generator=gen)

    loader_kw = dict(batch_size=batch_size, num_workers=0, pin_memory=True)
    train_loader = DataLoader(train_ds, shuffle=True,  **loader_kw)
    val_loader   = DataLoader(val_ds,   shuffle=False, **loader_kw)
    test_loader  = DataLoader(test_ds,  shuffle=False, **loader_kw)

    return train_loader, val_loader, test_loader


def train_model(model, train_loader, val_loader, adj_norm, device,
                save_path, lr=1e-3, max_epochs=100,
                patience_sched=5, patience_stop=10, verbose=True):
    """Train a model and return training history."""
    model = model.to(device)
    adj_norm = adj_norm.to(device)

    optimiser = torch.optim.Adam(model.parameters(), lr=lr)
    scheduler = torch.optim.lr_scheduler.ReduceLROnPlateau(
        optimiser, mode='min', factor=0.5, patience=patience_sched)

    best_val_loss = float('inf')
    epochs_no_improve = 0
    history = {'train_loss': [], 'val_loss': [], 'val_nmse_db_per_step': []}

    for epoch in range(1, max_epochs + 1):
        t0 = time.time()

        # Train
        model.train()
        train_losses = []
        for xb, yb in train_loader:
            xb, yb = xb.to(device), yb.to(device)
            optimiser.zero_grad()
            pred = model(xb, adj_norm)
            loss = nmse_loss(pred, yb)
            loss.backward()
            nn.utils.clip_grad_norm_(model.parameters(), max_norm=5.0)
            optimiser.step()
            train_losses.append(loss.item())

        # Validate
        model.eval()
        val_losses = []
        with torch.no_grad():
            for xb, yb in val_loader:
                xb, yb = xb.to(device), yb.to(device)
                pred = model(xb, adj_norm)
                val_losses.append(nmse_loss(pred, yb).item())

            all_preds, all_targets = [], []
            for xb, yb in val_loader:
                xb, yb = xb.to(device), yb.to(device)
                all_preds.append(model(xb, adj_norm))
                all_targets.append(yb)
            all_preds   = torch.cat(all_preds)
            all_targets = torch.cat(all_targets)

            T_predict = all_preds.shape[-1]
            per_step_nmse = [
                nmse_db(all_preds[..., k], all_targets[..., k])
                for k in range(T_predict)
            ]

        mean_train = float(torch.tensor(train_losses).mean())
        mean_val   = float(torch.tensor(val_losses).mean())
        history['train_loss'].append(mean_train)
        history['val_loss'].append(mean_val)
        history['val_nmse_db_per_step'].append(per_step_nmse)

        scheduler.step(mean_val)

        if mean_val < best_val_loss:
            best_val_loss = mean_val
            epochs_no_improve = 0
            os.makedirs(os.path.dirname(os.path.abspath(save_path)), exist_ok=True)
            torch.save(model.state_dict(), save_path)
        else:
            epochs_no_improve += 1

        if verbose and epoch % 5 == 0:
            step_str = '  '.join(
                [f"k{k+1}={ps:.2f}dB" for k, ps in enumerate(per_step_nmse)])
            print(f"Ep {epoch:3d}/{max_epochs} | "
                  f"train={mean_train:.4f}  val={mean_val:.4f} | "
                  f"{step_str} | "
                  f"lr={optimiser.param_groups[0]['lr']:.5f} | "
                  f"{time.time()-t0:.1f}s")

        if epochs_no_improve >= patience_stop:
            if verbose:
                print(f"Early stopping at epoch {epoch} "
                      f"(no val improvement for {patience_stop} epochs)")
            break

    history['best_val_nmse_linear'] = best_val_loss
    return history

## 5. Run: Generate All Datasets

In [6]:
print("Generating all datasets...")
generate_all_datasets(
    data_dir=DATA_DIR,
    L=16, K=4, N=4,
    fc=3.5e9,
    T_history=10,
    T_predict=3,
    num_samples=5000,
    Ts=1e-3,
)
print("Done generating datasets.")

Generating all datasets...

-- Generating: layout=random, velocity=30 km/h --
  Generating sample 0/5000...
  Generating sample 500/5000...
  Generating sample 1000/5000...
  Generating sample 1500/5000...
  Generating sample 2000/5000...
  Generating sample 2500/5000...
  Generating sample 3000/5000...
  Generating sample 3500/5000...
  Generating sample 4000/5000...
  Generating sample 4500/5000...
Saved -> /kaggle/working/data/dataset_random_30kmh.pt  |  X:torch.Size([5000, 16, 32, 10])  Y:torch.Size([5000, 16, 32, 3])

-- Generating: layout=grid, velocity=30 km/h --
  Generating sample 0/5000...
  Generating sample 500/5000...
  Generating sample 1000/5000...
  Generating sample 1500/5000...
  Generating sample 2000/5000...
  Generating sample 2500/5000...
  Generating sample 3000/5000...
  Generating sample 3500/5000...
  Generating sample 4000/5000...
  Generating sample 4500/5000...
Saved -> /kaggle/working/data/dataset_grid_30kmh.pt  |  X:torch.Size([5000, 16, 32, 10])  Y:torch

## 6. Run: Train All Models (12 configs)

In [7]:
MODEL_PARAMS = dict(L=16, N=4, K=4, T_history=10, T_predict=3, hidden_dim=64)
DS_PARAMS    = dict(L=16, K=4, N=4, fc=3.5e9, T_history=10, T_predict=3,
                    num_samples=5000)

VELOCITIES = [30, 60, 120]
LAYOUTS    = ['random', 'grid']

for model_type in ['gnn_cnn', 'cnn_only']:
    for layout in LAYOUTS:
        for vel in VELOCITIES:
            print(f"\n{'='*60}")
            print(f"  Training: {model_type.upper()} | layout={layout} | vel={vel} km/h")
            print(f"{'='*60}")

            ds = load_or_generate(layout, vel, data_dir=DATA_DIR, **DS_PARAMS)
            adj_norm = ds['adj_norm']

            train_loader, val_loader, _ = make_loaders(ds, batch_size=32)

            if model_type == 'gnn_cnn':
                model = GNNCNNHybrid(**MODEL_PARAMS)
            else:
                model = CNNOnly(**MODEL_PARAMS)

            print(f"  Parameters: {count_parameters(model):,}")

            ckpt = os.path.join(CKPT_DIR,
                                f'{model_type}_{layout}_{int(vel)}kmh.pt')

            history = train_model(
                model, train_loader, val_loader, adj_norm,
                device=DEVICE, save_path=ckpt,
                lr=1e-3, max_epochs=100,
            )
            print(f"  Best val NMSE (linear): {history['best_val_nmse_linear']:.5f}")
            print(f"  Checkpoint: {ckpt}")

print("\n\nAll training complete!")


  Training: GNN_CNN | layout=random | vel=30 km/h
Loading dataset from /kaggle/working/data/dataset_random_30kmh.pt
  Parameters: 51,008
Ep   5/100 | train=0.9618  val=0.9471 | k1=-0.23dB  k2=-0.30dB  k3=-0.23dB | lr=0.00100 | 0.7s
Ep  10/100 | train=0.8828  val=0.8939 | k1=-0.56dB  k2=-0.81dB  k3=-0.66dB | lr=0.00100 | 0.7s
Ep  15/100 | train=0.8613  val=0.8765 | k1=-0.49dB  k2=-0.80dB  k3=-0.73dB | lr=0.00100 | 0.7s
Ep  20/100 | train=0.8523  val=0.8766 | k1=-0.51dB  k2=-0.77dB  k3=-0.68dB | lr=0.00100 | 0.7s
Ep  25/100 | train=0.8451  val=0.8767 | k1=-0.44dB  k2=-0.71dB  k3=-0.65dB | lr=0.00050 | 0.7s
Ep  30/100 | train=0.8349  val=0.8722 | k1=-0.56dB  k2=-0.81dB  k3=-0.70dB | lr=0.00050 | 0.7s
Ep  35/100 | train=0.8289  val=0.8731 | k1=-0.55dB  k2=-0.92dB  k3=-0.86dB | lr=0.00050 | 0.7s
Ep  40/100 | train=0.8202  val=0.8723 | k1=-0.59dB  k2=-0.95dB  k3=-0.87dB | lr=0.00025 | 0.7s
Ep  45/100 | train=0.8152  val=0.8709 | k1=-0.59dB  k2=-0.96dB  k3=-0.89dB | lr=0.00013 | 0.7s
Ep  50/

## 7. Evaluation — All Experiments

In [8]:
# ── Helpers ──

PLOT_STYLES = {
    'AR':       dict(marker='s', linestyle='--', color='#e74c3c'),
    'Kalman':   dict(marker='^', linestyle='--', color='#e67e22'),
    'CNN-Only': dict(marker='o', linestyle='-',  color='#3498db'),
    'GNN-CNN':  dict(marker='D', linestyle='-',  color='#2ecc71', linewidth=2),
}


def load_model(model_type, layout, velocity, device):
    ckpt = os.path.join(CKPT_DIR,
                        f'{model_type}_{layout}_{int(velocity)}kmh.pt')
    if not os.path.exists(ckpt):
        return None
    mp = MODEL_PARAMS.copy()
    cls = GNNCNNHybrid if model_type == 'gnn_cnn' else CNNOnly
    model = cls(**mp).to(device)
    model.load_state_dict(torch.load(ckpt, map_location=device, weights_only=False))
    model.eval()
    return model


def get_test_loader(layout, velocity, batch_size=64):
    ds = load_or_generate(layout, velocity, data_dir=DATA_DIR, **DS_PARAMS)
    _, _, test_loader = make_loaders(ds, batch_size=batch_size)
    return test_loader, ds


def eval_nn_model(model, test_loader, adj_norm, device, k_step=None):
    model.eval()
    adj_norm = adj_norm.to(device)
    preds, targets = [], []
    with torch.no_grad():
        for xb, yb in test_loader:
            xb = xb.to(device)
            p  = model(xb, adj_norm).cpu()
            preds.append(p)
            targets.append(yb)
    preds   = torch.cat(preds)
    targets = torch.cat(targets)

    if k_step is not None:
        return nmse_db(preds[..., k_step], targets[..., k_step])
    return nmse_db(preds, targets)


def eval_baselines_on_dataset(layout, velocity, k_step=None):
    ds = load_or_generate(layout, velocity, data_dir=DATA_DIR, **DS_PARAMS)
    X, Y = ds['X'], ds['Y']

    S = len(X)
    n_train = int(S * 0.8)
    n_val   = int(S * 0.1)
    n_test  = S - n_train - n_val
    gen = torch.Generator().manual_seed(0)
    _, _, test_idx = random_split(
        list(range(S)), [n_train, n_val, n_test], generator=gen)
    idx = list(test_idx)

    N, K       = ds['N'], ds['K']
    T_predict  = ds['T_predict']
    fc, Ts     = ds['fc'], ds['Ts']

    X_cplx = flat_to_complex(X[idx], N, K)
    Y_cplx = flat_to_complex(Y[idx], N, K)

    ar = ARPredictor(order=4)
    kf = VectorKalmanPredictor(order=1)

    results = {}
    for name, pred_obj in [('AR', ar), ('VKF', kf)]:
        results[name] = evaluate_baseline(
            pred_obj, X_cplx, Y_cplx, T_predict, k_step=k_step)
    return results

### Experiment 1: NMSE vs Velocity [PRIMARY]

In [10]:
def exp_nmse_vs_velocity(layout='random'):
    print(f"\n{'='*55}")
    print(f" Experiment 1: NMSE vs Velocity  [layout={layout}]")
    print(f"{'='*55}")

    results = {m: [] for m in ['AR', 'VKF', 'CNN-Only', 'GNN-CNN']}

    for vel in VELOCITIES:
        print(f"\n  Velocity = {vel} km/h")

        bl = eval_baselines_on_dataset(layout, vel)
        results['AR'].append(bl['AR'])
        results['VKF'].append(bl['VKF'])
        print(f"    AR       : {bl['AR']:+.2f} dB")
        print(f"    VKF      : {bl['VKF']:+.2f} dB")

        test_loader, ds = get_test_loader(layout, vel)
        adj_norm = ds['adj_norm']

        for mtype, mkey in [('cnn_only', 'CNN-Only'), ('gnn_cnn', 'GNN-CNN')]:
            model = load_model(mtype, layout, vel, DEVICE)
            if model is None:
                results[mkey].append(None)
            else:
                val = eval_nn_model(model, test_loader, adj_norm, DEVICE)
                results[mkey].append(val)
                print(f"    {mkey:10s}: {val:+.2f} dB")

    json_path = os.path.join(RESULTS_DIR, f'nmse_vs_velocity_{layout}.json')
    with open(json_path, 'w') as f:
        json.dump({'velocities': VELOCITIES, 'results': results}, f, indent=2)

    # Plot
    fig, ax = plt.subplots(figsize=(8, 5))
    for name, vals in results.items():
        clean = [v if v is not None else np.nan for v in vals]
        ax.plot(VELOCITIES, clean, label=name,
                markersize=8, **PLOT_STYLES.get(name, {}))
    ax.set_xlabel('User Velocity (km/h)', fontsize=13)
    ax.set_ylabel('NMSE (dB)', fontsize=13)
    ax.set_title(f'NMSE vs User Velocity  [AP layout: {layout}]', fontsize=13)
    ax.set_xticks(VELOCITIES)
    ax.legend(fontsize=11)
    ax.grid(True, alpha=0.3)
    ax.invert_yaxis()
    path = os.path.join(RESULTS_DIR, f'fig1_nmse_vs_velocity_{layout}.png')
    fig.savefig(path, dpi=150, bbox_inches='tight')
    plt.show()
    print(f"  Figure saved -> {path}")
    return results


exp_nmse_vs_velocity('random')
exp_nmse_vs_velocity('grid')


 Experiment 1: NMSE vs Velocity  [layout=random]

  Velocity = 30 km/h
Loading dataset from /kaggle/working/data/dataset_random_30kmh.pt
    AR       : +16.65 dB
    VKF      : -3.67 dB
Loading dataset from /kaggle/working/data/dataset_random_30kmh.pt
    CNN-Only  : -10.79 dB
    GNN-CNN   : -1.16 dB

  Velocity = 60 km/h
Loading dataset from /kaggle/working/data/dataset_random_60kmh.pt
    AR       : +21.79 dB
    VKF      : -0.52 dB
Loading dataset from /kaggle/working/data/dataset_random_60kmh.pt
    CNN-Only  : -3.08 dB
    GNN-CNN   : -0.66 dB

  Velocity = 120 km/h
Loading dataset from /kaggle/working/data/dataset_random_120kmh.pt
    AR       : +0.54 dB
    VKF      : -0.48 dB
Loading dataset from /kaggle/working/data/dataset_random_120kmh.pt
    CNN-Only  : -0.49 dB
    GNN-CNN   : +0.03 dB
  Figure saved -> /kaggle/working/results/fig1_nmse_vs_velocity_random.png

 Experiment 1: NMSE vs Velocity  [layout=grid]

  Velocity = 30 km/h
Loading dataset from /kaggle/working/data/d

{'AR': [np.float64(24.25572545393645),
  np.float64(12.146240950979347),
  np.float64(24.844701046384237)],
 'VKF': [np.float64(-4.18500989184889),
  np.float64(-1.4403571872880057),
  np.float64(-0.903469299848229)],
 'CNN-Only': [-11.040464639663696, -2.99326628446579, -0.38591399788856506],
 'GNN-CNN': [-2.7608954906463623, -1.0686835646629333, -0.10458366014063358]}

### Experiment 2: NMSE vs Prediction Step

In [13]:
def exp_nmse_vs_step(layout='random', velocity=60.0):
    print(f"\n{'='*55}")
    print(f" Experiment 2: NMSE vs Prediction Step")
    print(f" layout={layout}  |  velocity={int(velocity)} km/h")
    print(f"{'='*55}")

    T_predict = MODEL_PARAMS['T_predict']
    results   = {m: [] for m in ['AR', 'VKF', 'CNN-Only', 'GNN-CNN']}

    test_loader, ds = get_test_loader(layout, velocity)
    adj_norm = ds['adj_norm']

    for k in range(T_predict):
        print(f"\n  k = {k+1}")
        bl = eval_baselines_on_dataset(layout, velocity, k_step=k)
        results['AR'].append(bl['AR'])
        results['VKF'].append(bl['VKF'])

        for mtype, mkey in [('cnn_only', 'CNN-Only'), ('gnn_cnn', 'GNN-CNN')]:
            model = load_model(mtype, layout, velocity, DEVICE)
            if model is None:
                results[mkey].append(None)
            else:
                val = eval_nn_model(model, test_loader, adj_norm, DEVICE, k_step=k)
                results[mkey].append(val)
                print(f"    {mkey:10s}: {val:+.2f} dB")

    json_path = os.path.join(RESULTS_DIR, f'nmse_vs_step_{layout}_{int(velocity)}.json')
    with open(json_path, 'w') as f:
        json.dump({'steps': list(range(1, T_predict + 1)),
                   'results': results}, f, indent=2)

    fig, ax = plt.subplots(figsize=(7, 5))
    x = list(range(1, T_predict + 1))
    for name, vals in results.items():
        clean = [v if v is not None else np.nan for v in vals]
        ax.plot(x, clean, label=name, markersize=8, **PLOT_STYLES.get(name, {}))
    ax.set_xlabel('Prediction Step k', fontsize=13)
    ax.set_ylabel('NMSE (dB)', fontsize=13)
    ax.set_title(f'NMSE vs Prediction Step [{layout}, {int(velocity)} km/h]', fontsize=13)
    ax.set_xticks(x)
    ax.legend(fontsize=11)
    ax.grid(True, alpha=0.3)
    ax.invert_yaxis()
    path = os.path.join(RESULTS_DIR, f'fig2_nmse_vs_step_{layout}_{int(velocity)}.png')
    fig.savefig(path, dpi=150, bbox_inches='tight')
    plt.show()
    return results


exp_nmse_vs_step('random', 60.0)
exp_nmse_vs_step('random', 120.0)


 Experiment 2: NMSE vs Prediction Step
 layout=random  |  velocity=60 km/h
Loading dataset from /kaggle/working/data/dataset_random_60kmh.pt

  k = 1
Loading dataset from /kaggle/working/data/dataset_random_60kmh.pt
    CNN-Only  : -8.48 dB
    GNN-CNN   : -0.96 dB

  k = 2
Loading dataset from /kaggle/working/data/dataset_random_60kmh.pt
    CNN-Only  : -2.94 dB
    GNN-CNN   : -0.82 dB

  k = 3
Loading dataset from /kaggle/working/data/dataset_random_60kmh.pt
    CNN-Only  : -1.09 dB
    GNN-CNN   : -0.29 dB

 Experiment 2: NMSE vs Prediction Step
 layout=random  |  velocity=120 km/h
Loading dataset from /kaggle/working/data/dataset_random_120kmh.pt

  k = 1
Loading dataset from /kaggle/working/data/dataset_random_120kmh.pt
    CNN-Only  : -0.63 dB
    GNN-CNN   : -0.09 dB

  k = 2
Loading dataset from /kaggle/working/data/dataset_random_120kmh.pt
    CNN-Only  : -0.65 dB
    GNN-CNN   : -0.16 dB

  k = 3
Loading dataset from /kaggle/working/data/dataset_random_120kmh.pt
    CNN-Onl

{'AR': [np.float64(-1.538937191006926),
  np.float64(0.9048695326927789),
  np.float64(1.6384941617183517)],
 'VKF': [np.float64(-1.2383674277080088),
  np.float64(-0.18698332953834065),
  np.float64(-0.10136350862233391)],
 'CNN-Only': [-0.6325674802064896, -0.6487278640270233, -0.22297328338027],
 'GNN-CNN': [-0.08715543895959854, -0.15727313235402107, 0.29825715348124504]}

### Experiment 3: Ablation -- CNN-Only vs GNN-CNN

In [14]:
def exp_ablation_gcn(layout='random'):
    print(f"\n{'='*55}")
    print(f" Experiment 3: Ablation -- GCN Contribution  [layout={layout}]")
    print(f"{'='*55}")

    cnn_vals, gnn_vals = [], []

    for vel in VELOCITIES:
        test_loader, ds = get_test_loader(layout, vel)
        adj_norm = ds['adj_norm']

        cnn_model = load_model('cnn_only', layout, vel, DEVICE)
        gnn_model = load_model('gnn_cnn',  layout, vel, DEVICE)

        cnn_v = eval_nn_model(cnn_model, test_loader, adj_norm, DEVICE) if cnn_model else None
        gnn_v = eval_nn_model(gnn_model, test_loader, adj_norm, DEVICE) if gnn_model else None

        cnn_vals.append(cnn_v)
        gnn_vals.append(gnn_v)

        if cnn_v is not None and gnn_v is not None:
            gain = gnn_v - cnn_v
            print(f"  {vel} km/h: CNN-Only={cnn_v:+.2f}dB  GNN-CNN={gnn_v:+.2f}dB  (GCN gain: {gain:+.2f}dB)")

    results = {'CNN-Only': cnn_vals, 'GNN-CNN': gnn_vals}

    json_path = os.path.join(RESULTS_DIR, f'ablation_gcn_{layout}.json')
    with open(json_path, 'w') as f:
        json.dump({'velocities': VELOCITIES, 'results': results}, f, indent=2)

    fig, ax = plt.subplots(figsize=(8, 5))
    x = np.arange(len(VELOCITIES))
    w = 0.35
    safe = lambda v: v if v is not None else 0.0
    ax.bar(x - w/2, [safe(v) for v in cnn_vals], w, label='CNN-Only', color='#3498db', alpha=0.85)
    ax.bar(x + w/2, [safe(v) for v in gnn_vals], w, label='GNN-CNN',  color='#2ecc71', alpha=0.85)
    for i, (c, g) in enumerate(zip(cnn_vals, gnn_vals)):
        if c is not None:
            ax.text(x[i]-w/2, c-0.3, f'{c:.1f}', ha='center', va='top', fontsize=9)
        if g is not None:
            ax.text(x[i]+w/2, g-0.3, f'{g:.1f}', ha='center', va='top', fontsize=9)
    ax.set_xlabel('User Velocity (km/h)', fontsize=13)
    ax.set_ylabel('NMSE (dB)', fontsize=13)
    ax.set_title(f'Ablation: GCN Contribution  [layout={layout}]', fontsize=13)
    ax.set_xticks(x)
    ax.set_xticklabels([f'{v} km/h' for v in VELOCITIES])
    ax.legend(fontsize=11)
    ax.grid(True, axis='y', alpha=0.3)
    ax.invert_yaxis()
    path = os.path.join(RESULTS_DIR, f'fig3_ablation_gcn_{layout}.png')
    fig.savefig(path, dpi=150, bbox_inches='tight')
    plt.show()
    return results


exp_ablation_gcn('random')
exp_ablation_gcn('grid')


 Experiment 3: Ablation -- GCN Contribution  [layout=random]
Loading dataset from /kaggle/working/data/dataset_random_30kmh.pt
  30 km/h: CNN-Only=-10.79dB  GNN-CNN=-1.16dB  (GCN gain: +9.63dB)
Loading dataset from /kaggle/working/data/dataset_random_60kmh.pt
  60 km/h: CNN-Only=-3.08dB  GNN-CNN=-0.66dB  (GCN gain: +2.42dB)
Loading dataset from /kaggle/working/data/dataset_random_120kmh.pt
  120 km/h: CNN-Only=-0.49dB  GNN-CNN=+0.03dB  (GCN gain: +0.52dB)

 Experiment 3: Ablation -- GCN Contribution  [layout=grid]
Loading dataset from /kaggle/working/data/dataset_grid_30kmh.pt
  30 km/h: CNN-Only=-11.04dB  GNN-CNN=-2.76dB  (GCN gain: +8.28dB)
Loading dataset from /kaggle/working/data/dataset_grid_60kmh.pt
  60 km/h: CNN-Only=-2.99dB  GNN-CNN=-1.07dB  (GCN gain: +1.92dB)
Loading dataset from /kaggle/working/data/dataset_grid_120kmh.pt
  120 km/h: CNN-Only=-0.39dB  GNN-CNN=-0.10dB  (GCN gain: +0.28dB)


{'CNN-Only': [-11.040464639663696, -2.99326628446579, -0.38591399788856506],
 'GNN-CNN': [-2.7608954906463623, -1.0686835646629333, -0.10458366014063358]}

### Experiment 4: Ablation -- Grid vs Random AP Layout

In [15]:
def exp_ablation_topology(velocity=60.0):
    print(f"\n{'='*55}")
    print(f" Experiment 4: Topology Ablation  [v={int(velocity)} km/h]")
    print(f"{'='*55}")

    results = {}
    for lay in LAYOUTS:
        test_loader, ds = get_test_loader(lay, velocity)
        adj_norm = ds['adj_norm']
        model = load_model('gnn_cnn', lay, velocity, DEVICE)
        if model is None:
            results[lay] = None
        else:
            val = eval_nn_model(model, test_loader, adj_norm, DEVICE)
            results[lay] = val
            print(f"  {lay}: GNN-CNN NMSE = {val:+.2f} dB")

    json_path = os.path.join(RESULTS_DIR, f'ablation_topology_{int(velocity)}.json')
    with open(json_path, 'w') as f:
        json.dump({'layouts': LAYOUTS, 'velocity': velocity, 'results': results}, f, indent=2)

    fig, ax = plt.subplots(figsize=(5, 5))
    vals   = [results.get(l) or 0.0 for l in LAYOUTS]
    colors = ['#9b59b6', '#f39c12']
    bars   = ax.bar(LAYOUTS, vals, color=colors, alpha=0.85, width=0.4)
    for bar, v in zip(bars, vals):
        if v != 0.0:
            ax.text(bar.get_x() + bar.get_width()/2, v-0.2, f'{v:.2f} dB',
                    ha='center', va='top', fontsize=11, fontweight='bold')
    ax.set_ylabel('NMSE (dB)', fontsize=13)
    ax.set_title(f'Topology Ablation  [GNN-CNN, {int(velocity)} km/h]', fontsize=13)
    ax.grid(True, axis='y', alpha=0.3)
    ax.invert_yaxis()
    ax.set_xticklabels(['Random AP Layout', 'Grid AP Layout'], fontsize=11)
    path = os.path.join(RESULTS_DIR, f'fig4_ablation_topology_{int(velocity)}.png')
    fig.savefig(path, dpi=150, bbox_inches='tight')
    plt.show()
    return results


exp_ablation_topology(60.0)
exp_ablation_topology(120.0)


 Experiment 4: Topology Ablation  [v=60 km/h]
Loading dataset from /kaggle/working/data/dataset_random_60kmh.pt
  random: GNN-CNN NMSE = -0.66 dB
Loading dataset from /kaggle/working/data/dataset_grid_60kmh.pt
  grid: GNN-CNN NMSE = -1.07 dB

 Experiment 4: Topology Ablation  [v=120 km/h]
Loading dataset from /kaggle/working/data/dataset_random_120kmh.pt


/tmp/ipykernel_55/607022843.py:34: UserWarning: set_ticklabels() should only be used with a fixed number of ticks, i.e. after set_ticks() or using a FixedLocator.
  ax.set_xticklabels(['Random AP Layout', 'Grid AP Layout'], fontsize=11)


  random: GNN-CNN NMSE = +0.03 dB
Loading dataset from /kaggle/working/data/dataset_grid_120kmh.pt
  grid: GNN-CNN NMSE = -0.10 dB


/tmp/ipykernel_55/607022843.py:34: UserWarning: set_ticklabels() should only be used with a fixed number of ticks, i.e. after set_ticks() or using a FixedLocator.
  ax.set_xticklabels(['Random AP Layout', 'Grid AP Layout'], fontsize=11)


{'random': 0.026378808543086052, 'grid': -0.10458366014063358}

### Experiment 5: Efficiency Metrics

In [19]:
def exp_efficiency(layout='random', velocity=60.0):
    print(f"\n{'='*55}")
    print(f" Experiment 5: Efficiency Metrics")
    print(f"{'='*55}")

    mp = MODEL_PARAMS.copy()
    ds = load_or_generate(layout, velocity, data_dir=DATA_DIR, **DS_PARAMS)
    adj_norm = ds['adj_norm'].to(DEVICE)

    x_dummy = torch.randn(1, mp['L'], 2 * mp['N'] * mp['K'],
                          mp['T_history']).to(DEVICE)

    results = {}

    for mtype, label in [('gnn_cnn', 'GNN-CNN'), ('cnn_only', 'CNN-Only')]:
        model = load_model(mtype, layout, velocity, DEVICE)
        if model is None:
            cls = GNNCNNHybrid if mtype == 'gnn_cnn' else CNNOnly
            model = cls(**mp).to(DEVICE)

        n_params = count_parameters(model)
        mean_ms, std_ms = measure_latency(model, x_dummy, adj_norm, n_runs=200)

        results[label] = {
            'params': n_params,
            'latency_mean_ms': round(mean_ms, 3),
            'latency_std_ms':  round(std_ms, 3),
        }
        print(f"\n  {label}")
        print(f"    Parameters : {n_params:,}")
        print(f"    Latency    : {mean_ms:.3f} +/- {std_ms:.3f} ms")

    # Baselines
    N, K, T_h, T_p = mp['N'], mp['K'], mp['T_history'], mp['T_predict']
    L = mp['L']
    fc, Ts = ds['fc'], ds['Ts']
    x_np = np.random.randn(L, N, K, T_h) + 1j * np.random.randn(L, N, K, T_h)

    for bl_cls, label, extra_kw in [
        (ARPredictor,           'AR',     dict(order=4)),
        (VectorKalmanPredictor, 'VKF',    dict(order=1)),
    ]:
        predictor = bl_cls(**extra_kw)
        for _ in range(3):
            predictor.predict(x_np, T_p)
        times = []
        for _ in range(50):
            t0 = time.perf_counter()
            predictor.predict(x_np, T_p)
            times.append((time.perf_counter() - t0) * 1000)
        mean_ms = float(np.mean(times))
        std_ms  = float(np.std(times))
        param_equiv = (L * N * K * 4) if label == 'AR' else (L * N * K)
        results[label] = {
            'params': param_equiv,
            'latency_mean_ms': round(mean_ms, 3),
            'latency_std_ms':  round(std_ms, 3),
        }
        print(f"\n  {label}")
        print(f"    Param-equiv: {param_equiv:,}")
        print(f"    Latency    : {mean_ms:.3f} +/- {std_ms:.3f} ms")

    json_path = os.path.join(RESULTS_DIR, 'efficiency.json')
    with open(json_path, 'w') as f:
        json.dump(results, f, indent=2)

    # Summary table
    print(f"\n{'─'*62}")
    print(f"  {'Method':<12} {'Parameters':>12}  {'Latency (ms)':>16}  Edge?")
    print(f"{'─'*62}")
    for method, info in results.items():
        p   = info['params']
        lat = info['latency_mean_ms']
        std = info['latency_std_ms']
        edge = 'Y' if lat < 5.0 else 'N'
        p_str = f"{p:,}" if p < 1_000_000 else f"{p/1e6:.2f}M"
        print(f"  {method:<12} {p_str:>12}  {lat:>8.3f} +/- {std:<5.3f}   {edge}")
    print(f"{'─'*62}")
    return results


exp_efficiency('random', 60.0)


 Experiment 5: Efficiency Metrics
Loading dataset from /kaggle/working/data/dataset_random_60kmh.pt

  GNN-CNN
    Parameters : 51,008
    Latency    : 0.928 +/- 0.146 ms

  CNN-Only
    Parameters : 46,848
    Latency    : 0.712 +/- 0.038 ms

  AR
    Param-equiv: 1,024
    Latency    : 9.639 +/- 0.359 ms

  VKF
    Param-equiv: 256
    Latency    : 35.972 +/- 1.037 ms

──────────────────────────────────────────────────────────────
  Method         Parameters      Latency (ms)  Edge?
──────────────────────────────────────────────────────────────
  GNN-CNN            51,008     0.928 +/- 0.146   Y
  CNN-Only           46,848     0.712 +/- 0.038   Y
  AR                  1,024     9.639 +/- 0.359   N
  VKF                   256    35.972 +/- 1.037   N
──────────────────────────────────────────────────────────────


{'GNN-CNN': {'params': 51008,
  'latency_mean_ms': 0.928,
  'latency_std_ms': 0.146},
 'CNN-Only': {'params': 46848,
  'latency_mean_ms': 0.712,
  'latency_std_ms': 0.038},
 'AR': {'params': 1024, 'latency_mean_ms': 9.639, 'latency_std_ms': 0.359},
 'VKF': {'params': 256, 'latency_mean_ms': 35.972, 'latency_std_ms': 1.037}}

### Summary Table

In [20]:
# Print final summary for both layouts
for layout in ['random', 'grid']:
    json_path = os.path.join(RESULTS_DIR, f'nmse_vs_velocity_{layout}.json')
    if not os.path.exists(json_path):
        continue
    with open(json_path) as f:
        data = json.load(f)

    print(f"\n{'='*65}")
    print(f" NMSE Summary Table  [layout={layout}]  (dB, lower is better)")
    print(f"{'='*65}")
    header = f"  {'Method':<12}" + "".join(f"  {v:>6} km/h" for v in VELOCITIES)
    print(header)
    print(f"{'─'*65}")
    for method, vals in data['results'].items():
        row = f"  {method:<12}"
        for v in vals:
            row += f"  {v:>10.2f}" if v is not None else f"  {'N/A':>10}"
        print(row)
    print(f"{'─'*65}")


 NMSE Summary Table  [layout=random]  (dB, lower is better)
  Method            30 km/h      60 km/h     120 km/h
─────────────────────────────────────────────────────────────────
  AR                 16.65       21.79        0.54
  VKF                -3.67       -0.52       -0.48
  CNN-Only          -10.79       -3.08       -0.49
  GNN-CNN            -1.16       -0.66        0.03
─────────────────────────────────────────────────────────────────

 NMSE Summary Table  [layout=grid]  (dB, lower is better)
  Method            30 km/h      60 km/h     120 km/h
─────────────────────────────────────────────────────────────────
  AR                 24.26       12.15       24.84
  VKF                -4.19       -1.44       -0.90
  CNN-Only          -11.04       -2.99       -0.39
  GNN-CNN            -2.76       -1.07       -0.10
─────────────────────────────────────────────────────────────────


## Done!
All results are saved in `/kaggle/working/results/`

In [21]:
print("\nAll experiments complete!")
print(f"Results directory: {RESULTS_DIR}")
for f in sorted(os.listdir(RESULTS_DIR)):
    size = os.path.getsize(os.path.join(RESULTS_DIR, f))
    print(f"  {f:<45s}  {size:>8,} bytes")


All experiments complete!
Results directory: /kaggle/working/results
  ablation_gcn_grid.json                              277 bytes
  ablation_gcn_random.json                            277 bytes
  ablation_topology_120.json                          157 bytes
  ablation_topology_60.json                           154 bytes
  efficiency.json                                     388 bytes
  fig1_nmse_vs_velocity_grid.png                   66,645 bytes
  fig1_nmse_vs_velocity_random.png                 64,929 bytes
  fig2_nmse_vs_step_random_120.png                 69,208 bytes
  fig2_nmse_vs_step_random_60.png                  58,970 bytes
  fig3_ablation_gcn_grid.png                       40,487 bytes
  fig3_ablation_gcn_random.png                     41,615 bytes
  fig4_ablation_topology_120.png                   42,061 bytes
  fig4_ablation_topology_60.png                    33,657 bytes
  nmse_vs_step_random_120.json                        466 bytes
  nmse_vs_step_random_60.json     